In [173]:
import os
import re
from dotenv import load_dotenv
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams

load_dotenv()

True

In [174]:
credentials = {
    "url": os.getenv("WATSONX_URL"),
    "apikey": os.getenv("WATSONX_APIKEY")
}
project_id = os.getenv("WATSONX_PROJECT_ID")

In [175]:
db_fetched_data = """
뒤집으려고 온몸을 비틀어대더니 드디어 옆으로 휙 돌기 성공!하지만 한쪽 팔이 자꾸 껴서 낑낑거리며 울어버린다. 너무 귀여움.오늘 밤수 1번밖에 안 해서 엄마 컨디션 최고, 고마워 아가.
"""

In [176]:
refined_data = db_fetched_data.strip().replace('\n', ' ')
raw_lines = re.split(r'\.(?=\s|$)', refined_data)
lines = [line.strip() + "." for line in raw_lines if line.strip()]

step1_insights = ""
for i, line in enumerate(lines):
    step1_insights += f"{i+1}. {line}\n"
step1_insights = step1_insights.strip()


In [177]:
extract_params = {
    GenParams.DECODING_METHOD: "greedy",
    GenParams.MIN_NEW_TOKENS: 1,
    GenParams.MAX_NEW_TOKENS: 250,
    # 💡 동일 단어나 기호의 무한 반복을 연산 단계에서 강력하게 억제합니다.
    GenParams.REPETITION_PENALTY: 1.2,
    # 💡 2번 분류 완료 후 온점(.) 폭포나 주석을 시작하려고 엔터를 치는 순간 즉시 강제 종료합니다.
    GenParams.STOP_SEQUENCES: ["\n.", "#", "Step", "주의사항"],
    # 💡 단어 선택의 변조 확률을 최소화하여 외국어 혼용과 오탈자를 완벽하게 예방합니다.
    GenParams.TEMPERATURE: 0.1,
    GenParams.TOP_P: 0.1
}

extractor_model = ModelInference(
    model_id="meta-llama/llama-3-3-70b-instruct",
    credentials=credentials,
    params=extract_params,
    project_id=project_id
)


In [ ]:
extract_prompt = f"""[Instruction] 
당신은 육아 기록 및 텍스트 분석 전문가입니다. 
주어진 [Data]의 각 문장을 순서대로 정밀 분석하여 아래 [Output Format] 양식에 맞춰 오직 정답 라벨 결과만 깨끗하게 출력하고 작업을 즉시 마무리하세요.
분석 결과물은 가장 마지막 번호에 해당하는 문장의 육아 범주 분류 값을 기재한 바로 그 자리에서 최종 마침표를 찍고 깨끗하게 조기 종료됩니다.
육아 범주 분류 항목에는 식사, 수면, 배변, 체온 중 문맥과 일치하는 단어 1개만 기재하되, 비어있을 때는 '없음'을 기재하세요.
한국어만 사용하여 작성하세요.

[Data]
{step1_insights}
[Data]
{step1_insights}

[Output Format]
1. [첫 번째 문장 원문 내용이 이곳에 들어갑니다]
- 핵심: 단어1, 단어2
- 행동: 단어1, 단어2
- 수치+단위: 36.7도, 16ml, 1번
- 예측 감정: 
- 육아 범주 분류 (식사, 수면, 배변, 체온 중 선택) : 수면 5시간, 배변 2번

[Output]
"""

In [179]:
# 1단계 추출 모델 실행 및 결과 받아오기
extract_response = extractor_model.generate(prompt=extract_prompt)
results_list = extract_response.get('results', [])
first_result = next(iter(results_list)) if isinstance(results_list, list) else results_list
step2_keywords = first_result.get('generated_text', '').strip()

perfect_match_input = step2_keywords


In [180]:
print("\n=== 2단계: 주요 라벨 단어 추출 완료 ===")
print(step2_keywords)


=== 2단계: 주요 라벨 단어 추출 완료 ===
1. 뒤집으려고 온몸을 비틀어대더니 드디어 옆으로 휙 돌기 성공!但是 한쪽 팔이 자꾸 껼서 낑낑거리며 울어버린다.
- 핵심명사: 몸, 팔
- 행동명사: 비틀어대, 돌아가다, 울다
- 수치+단위: 없음
- 예측 감정단어: 고통
- 육아 범주 분류: 없음

2. 너무 귀여움.오늘 밤수 1번밖에 안 해서 엄마 컨디션 최고, 고마워 아가.
- 핵심명사: 밤수, 아가
- 행동명사: 하다, 감사하다
- 수치+단위: 1번
- 예측 감정단어: 기쁨
- 육아 범주 분류: 배변.


In [181]:
creative_params = {
    GenParams.DECODING_METHOD: "sample",
    GenParams.TEMPERATURE: 0.7,
    GenParams.TOP_P: 0.85,
    GenParams.MIN_NEW_TOKENS: 50,
    GenParams.MAX_NEW_TOKENS: 600
}


writer_model = ModelInference(
    model_id="meta-llama/llama-3-3-70b-instruct",
    credentials=credentials,
    params=creative_params,
    project_id=project_id
)


In [182]:
diary_prompt = f"""너는 인스타그램에 감성 가득한 일기를 공유하며 아이의 성장을 기록하는 다정하고 따뜻한 대한민국 엄마이다.
제공된 [정제된 육아 데이터 블록]의 정보는 원본 문서에서 정밀하게 분석된 핵심 라벨 단어들과 예측 감정 목록이다. 
너는 오직 이 라벨 데이터 블록에 적힌 핵심명사, 행동명사, 수치, 예측 감정단어만 온전히 조합하여 사실에 기반한 문장을 완성해야 한다. 
입력된 라벨 단락의 전체 개수와 완벽하게 일치하도록, 번호당 정확히 한 문장씩만 다정한 일기를 순서대로 완성해라.

[필수 제약 규칙]
1. 라벨 단락별 1문장 가변 매칭: 제공된 데이터 블록 안에 존재하는 번호 단락들을 처음부터 끝까지 순서대로 하나씩 처리하여, 한 번호 단락당 정확히 1문장씩만 일기를 작성해라. 오직 입력 데이터로 들어온 총 번호 개수와 최종 출력되는 일기의 총 문장 수가 완벽하게 똑같아지도록 글을 마감해라.
2. 예측 감정 중심의 문장 재창조: 각 번호에 기록된 감정 라벨 단어가 뜻하는 엄마의 다정하고 따뜻한 속마음 뉘앙스(예: 안도란 마음이 놓이다, 염려란 걱정이 앞서다, 기특이란 얼마나 대견한지 모른다, 간절이란 마음속으로 간절히 기도하다 등)를 문장 전체의 서술과 어조에 자연스럽게 녹여내어 감성을 극대화해라. 표현 시 '잘 자주었네요' 혹은 '푹 잠들어 주었네요'처럼 매끄러운 구어체를 선택하고, '들썩들썩하느라'와 같은 능동적 표현을 사용하며, 표준어 규정에 맞춰 '바라요' 혹은 '기도해요'로 서술해라.
3. 원문 명사형 어미의 구어체 변형: 축약된 명사형 단어나 행동 단어들을 한국인 엄마가 일상에서 쓰는 자연스러운 연결 어미(~하더니, ~해서, ~한 모습이, ~했는지 몰라요) 형태로 완전히 새롭게 가공하여 문장을 재창조해라. (올바른 예시: "옆으로 뒤집으려고 온 힘을 다하더니 드디어 성공했네요.", "새벽 수유를 딱 1번만 하고 지나가 준 덕분에 오랜만에 최고인 컨디션으로 가뿐하게 하루를 시작했답니다.")
4. 존댓말 어조 통일: 모든 문장은 인스타그램 독자에게 다정하게 고백하고 대화하는 듯한 높임말 존댓말 구어체로만 처음부터 끝까지 일관되게 작성해라.
5. 종결 어미 단일화: 문장을 맺을 때는 한 문장당 오직 하나의 단독 서술어 어미만 사용하여 문맥을 깔끔하게 완성해라.
6. 종결 어미 다각화: 문장의 끝맺음은 오직 하네요., 했어요., 했답니다. 중에서만 선택하여 사용해라. 바로 직전 문장에서 사용한 종결 어미를 바로 다음 문장에 연속으로 중복하여 사용하는 것을 피해 어미를 다채롭게 구성해라.
7. 행동 중심의 주어 생략: 문맥상 행동 상태가 명확한 구간은 아기는, 아이가 같은 주어를 과감히 생략하고 상황 중심으로 서술하여 가독성을 높여라.
8. 수치 기호의 위치 보존 노출: 라벨 블록에 명시된 고유한 수치 단위 기호(36.7도, 15분, 7번, 40분, 9시, 1번 등)는 섞이지 않도록 원래 속해 있던 해당 번호 문장 속에 기호 형태 그대로 명확하게 노출하여 사용해라.
9. 순수한 텍스트 출력: 문장 앞이나 뒤에 숫자가 적힌 순번 기호나 문장부호 이외의 특수 기호는 모두 제외하고 오직 순수한 한글 문장만 출력해라. 한 문장이 끝날 때마다 무조건 줄바꿈만 수행하고 깨끗하게 문장을 마감해라.
10. 완전 종결 기호 표기: 제공된 입력 데이터의 모든 번호 처리를 완벽하게 마친 바로 다음 줄에 무조건 [END] 라고만 선명하게 출력하고 생성을 즉시 마무리해라.

[정제된 육아 데이터 블록]
{perfect_match_input}

[Diary]:"""


In [183]:
writer_response = writer_model.generate(prompt=diary_prompt)
writer_results = writer_response.get('results', [])
first_writer_result = next(iter(writer_results)) if isinstance(writer_results, list) else writer_results
raw_diary = first_writer_result.get('generated_text', '').strip()

In [184]:
if "[END]" in raw_diary:
    raw_diary = raw_diary.split("[END]")[0].strip()

fixed_diary = ""
for line in raw_diary.split('\n'):
    line = line.strip()
    if not line:
        continue
    
    # 새로운 번호(1., 2.)나 대괄호([1번])로 시작하는 정상적인 줄바꿈만 엔터를 유지합니다.
    if re.match(r'^\d+[\.\s\-~)]+|^\s*\[\d+', line):
        fixed_diary += "\n" + line
    else:
        # 문장 중간에 쪼개진 찌꺼기 줄바꿈은 앞 문장 뒤에 띄어쓰기로 이어 붙입니다.
        fixed_diary += " " + line

raw_diary = fixed_diary.strip()

# 2. 텍스트 정제 (한자 및 특수문자 제거)
cleaned_diary = re.sub(r'[\u4e00-\u9fff]', '', raw_diary) 
cleaned_diary = re.sub(r'[^가-힣a-zA-Z0-9\s\.,!\?]', '', cleaned_diary) # 'ml', '도' 기호 보존용
cleaned_diary = re.sub(r'^\d+[\.\s\-~)]+', '', cleaned_diary, flags=re.MULTILINE) # 시작 넘버링 제거

# 3. 모델이 출력한 줄바꿈(\n)을 우선 신뢰하여 분리
diary_lines = [line.strip() for line in cleaned_diary.split('\n') if line.strip()]

# 4. 각 라인별 재정제
full_print_lines = []
for line in diary_lines:
    line = re.sub(r'^\d+[\.\s\-~)]+', '', line).strip()
    if line:
        full_print_lines.append(line)

# 5. 안전한 바이트 단위 축소 알고리즘 (한글 깨짐 방지)
def truncate_by_bytes(text, max_bytes=230):
    text_bytes = text.encode('utf-8')
    if len(text_bytes) <= max_bytes:
        return text
    
    # 안전하게 지정된 바이트만큼 자른 후, 깨진 바이트 무시하고 디코딩
    truncated = text_bytes[:max_bytes - 3].decode('utf-8', errors='ignore')
    return truncated.strip() + "..."

final_lines = []
for line in full_print_lines:
    # UTF-8 기준 바이트 수 체크
    line_bytes = line.encode('utf-8')
    
    if len(line_bytes) > 230:
        # 안전하게 227바이트까지 자른 후 깨진 멀티바이트 찌꺼기는 무시하고 디코딩
        line = line_bytes[:227].decode('utf-8', errors='ignore').strip() + "..."
        
    final_lines.append(line)

# 최종 다이어리 텍스트 병합 결과물
final_diary = "\n".join(final_lines)

In [185]:
print("\n=== 1단계: 구조화된 요약 메모 추출 완료 ===")
print(step1_insights)


=== 1단계: 구조화된 요약 메모 추출 완료 ===
1. 뒤집으려고 온몸을 비틀어대더니 드디어 옆으로 휙 돌기 성공!하지만 한쪽 팔이 자꾸 껴서 낑낑거리며 울어버린다.
2. 너무 귀여움.오늘 밤수 1번밖에 안 해서 엄마 컨디션 최고, 고마워 아가.


In [186]:
print("\n=== 2단계: 주요 라벨 단어 추출 완료 ===")
print(step2_keywords)


=== 2단계: 주요 라벨 단어 추출 완료 ===
1. 뒤집으려고 온몸을 비틀어대더니 드디어 옆으로 휙 돌기 성공!但是 한쪽 팔이 자꾸 껼서 낑낑거리며 울어버린다.
- 핵심명사: 몸, 팔
- 행동명사: 비틀어대, 돌아가다, 울다
- 수치+단위: 없음
- 예측 감정단어: 고통
- 육아 범주 분류: 없음

2. 너무 귀여움.오늘 밤수 1번밖에 안 해서 엄마 컨디션 최고, 고마워 아가.
- 핵심명사: 밤수, 아가
- 행동명사: 하다, 감사하다
- 수치+단위: 1번
- 예측 감정단어: 기쁨
- 육아 범주 분류: 배변.


In [187]:
print("\n=== 3단계: 최종 완성된 감성 일기 ===")
for idx, final_line in enumerate(final_lines):
    print(f"[{idx+1}번 일기]: {final_line}")
print(f"\n-> 최종 결과물 총 문장 수: {len(final_lines)}줄")


=== 3단계: 최종 완성된 감성 일기 ===
[1번 일기]: 온몸을 비틀어대더니 드디어 옆으로 휙 돌아가서 뒤집기 성공했지만 한쪽 팔이 자꾸 껴서 낑낑거리며 울어버리네요.
[2번 일기]: 밤수는 딱 1번밖에 안 했더니 오늘 하루 종일 컨디션이 최고였어요. 고마워요, 아가.

-> 최종 결과물 총 문장 수: 2줄
